# Query Degradation — realistic-query robustness experiment

**Why:** MultiHopRAG's queries are LLM-generated *from* the source articles,
so they inherit exact proper nouns and precise phrasing. That hands BM25 an
unrealistic lexical-overlap advantage (the "question" is nearly a paraphrase
of the answer passage). This likely explains why BM25 beat dense.

**Experiment:** rewrite every non-null query to be shorter, more casual, and
with softened/dropped proper nouns — while keeping the **same information need
and the same gold evidence docs**. Then re-run BM25 + dense on the degraded
set. Hypothesis: as lexical overlap drops, BM25 degrades faster than dense;
the gap closes or flips.

**Controlled comparison:** identical gold labels (`evidence_list`), only the
query phrasing changes. We keep each degraded query paired with its original.

**Honest caveat:** LLM-degraded queries are a *proxy* for real user queries,
not actual traffic. The finding is directional ("less lexical overlap shrinks
BM25's edge"), an illustration, not a production claim.

**Prereqs:** `rag_eval/` uploaded next to this notebook. Runs in Studio
(exec role, no profile). Claude via the cross-region inference profile.

## 1. Session + load queries from S3

All non-null queries (those with gold evidence). Full set (~2,255) — no
sampling, so there's no sample-size caveat in the writeup.

In [ ]:
import sys; sys.path.insert(0, ".")
import io, json, time
import boto3, pandas as pd

REGION = "us-east-1"
BUCKET = "rag-arch-eval-data-875699230934"
DATA_PREFIX = "data"
OUT_KEY = "data/degraded_queries.parquet"
CKPT_KEY = "data/degraded_queries_checkpoint.parquet"  # incremental safety
# Amazon Nova Lite: Amazon-owned, no Anthropic use-case agreement gate.
# (Claude is blocked on agreementAvailability=NOT_AVAILABLE; Nova works now
# and is more than adequate for query rewriting.)
CHAT_MODEL = "us.amazon.nova-lite-v1:0"

s3 = boto3.client("s3", region_name=REGION)
brt = boto3.client("bedrock-runtime", region_name=REGION)

def read_parquet_s3(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))

q_df = read_parquet_s3(f"{DATA_PREFIX}/queries.parquet")
# keep non-null (has evidence); preserve evidence_list untouched
rows = []
for i, r in enumerate(q_df.itertuples()):
    rel = tuple(dict.fromkeys(e["title"] for e in r.evidence_list))
    if rel:
        rows.append({"qid": i, "original_query": r.query,
                     "question_type": r.question_type,
                     "evidence_titles": list(rel)})
base = pd.DataFrame(rows)
print("non-null queries to degrade:", len(base))
print(base["question_type"].value_counts().to_string())

## 2. The degradation prompt (mid aggressiveness)

The constraints matter: reduce lexical overlap (shorten, casual, soften/drop
proper nouns) but KEEP the question answerable from the same evidence — don't
change *what* is asked. We return only the rewritten question.

In [ ]:
SYSTEM = (
    "You rewrite search questions to sound like a real, hurried user typing into "
    "a search box. Make the question SHORTER, more casual, and less precise. "
    "Soften or drop exact proper nouns, exact dates, and exact titles (e.g. "
    "'the Verizon Q3 2023 earnings report' -> 'that carrier's recent earnings'). "
    "IMPORTANT: a real user does NOT know or cite which news outlet reported a "
    "story, so REMOVE references to the publication/source (e.g. drop 'as "
    "reported by The Verge and TechCrunch' entirely) — a real person just asks "
    "the question. BUT if the question is fundamentally ABOUT a source (e.g. "
    "'which outlet reported X'), keep what's needed so it stays answerable.\n",
    "CRITICAL: do NOT change what is being asked — the rewritten question must "
    "still be answerable from exactly the same source documents, just phrased "
    "more vaguely and naturally. Keep it a real question. Output ONLY the "
    "rewritten question, no quotes, no preamble."
)

def degrade(query: str) -> str:
    # Amazon Nova schema: system is a list of {text}; content is a list of
    # {text}; generation controls live under inferenceConfig. Response text is
    # at output.message.content[0].text (different from Claude's content[0].text).
    body = json.dumps({
        "system": [{"text": SYSTEM}],
        "messages": [{"role": "user", "content": [{"text": f"Rewrite this question:\n{query}"}]}],
        "inferenceConfig": {"maxTokens": 150, "temperature": 0.7},
    })
    resp = brt.invoke_model(modelId=CHAT_MODEL, contentType="application/json",
                            accept="application/json", body=body)
    payload = json.loads(resp["body"].read())
    return payload["output"]["message"]["content"][0]["text"].strip()

# smoke test on 3 examples before the full run
for q in base["original_query"].head(3):
    print("ORIG:", q)
    print("DEGR:", degrade(q))
    print("-" * 60)

## 3. (manual) Spot-check the smoke-test output

**Before the full run:** eyeball the 3 examples above. Each degraded query
should still be answerable from the same articles — vaguer, not *different*.
If a rewrite changed the actual question or became unanswerable, tighten the
SYSTEM prompt and re-run cell 2. Only proceed when they look right.

## 4. Full run with checkpointing

~2,255 sequential Nova calls — expect ~30-60 min. We checkpoint to S3 every
`CKPT_EVERY` queries so a kernel death doesn't lose work (and re-running this
cell resumes from the checkpoint).

In [ ]:
CKPT_EVERY = 100

# resume if a checkpoint exists
try:
    done = read_parquet_s3(CKPT_KEY)
    done_qids = set(done["qid"])
    results = done.to_dict("records")
    print(f"resuming: {len(done_qids)} already done")
except s3.exceptions.NoSuchKey:
    done_qids, results = set(), []
    print("fresh run")

def checkpoint():
    buf = io.BytesIO(); pd.DataFrame(results).to_parquet(buf, index=False); buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=CKPT_KEY, Body=buf.getvalue())

t0 = time.time()
for n, row in enumerate(base.itertuples(), start=1):
    if row.qid in done_qids:
        continue
    try:
        dq = degrade(row.original_query)
    except Exception as e:
        dq = None; print(f"  qid {row.qid} error: {str(e)[:80]}")
    results.append({"qid": row.qid, "question_type": row.question_type,
                    "original_query": row.original_query, "degraded_query": dq,
                    "evidence_titles": row.evidence_titles})
    if len(results) % CKPT_EVERY == 0:
        checkpoint()
        print(f"  {len(results)}/{len(base)}  ({time.time()-t0:.0f}s)")

checkpoint()
print(f"done: {len(results)} degraded in {time.time()-t0:.0f}s")

## 5. Finalize: write degraded_queries.parquet

Promote the checkpoint to the final artifact the eval notebooks read. Drop
any rows where degradation errored out (dq is None).

In [ ]:
final = pd.DataFrame(results)
n_err = final["degraded_query"].isna().sum()
final = final.dropna(subset=["degraded_query"]).reset_index(drop=True)
print(f"final degraded queries: {len(final)}  (dropped {n_err} errored)")

buf = io.BytesIO(); final.to_parquet(buf, index=False); buf.seek(0)
s3.put_object(Bucket=BUCKET, Key=OUT_KEY, Body=buf.getvalue())
print(f"wrote s3://{BUCKET}/{OUT_KEY}")

# quick before/after readout
import numpy as np
final["orig_words"] = final["original_query"].str.split().str.len()
final["degr_words"] = final["degraded_query"].str.split().str.len()
print("\nmedian words  original:", int(final["orig_words"].median()),
      " degraded:", int(final["degr_words"].median()))
final[["original_query", "degraded_query"]].head(5)

## 6. Measure lexical overlap drop (the manipulation check)

Quantify what the degradation actually did to BM25's signal: for each query,
what fraction of its content tokens appear in the concatenated GOLD documents?
We use the SAME tokenizer BM25 uses, so this is literally the lexical signal
BM25 keys on. If degradation worked, mean overlap drops original -> degraded.
This turns 'we made queries vaguer' into a measured variable, and explains any
BM25 degradation mechanistically.

In [ ]:
from rag_eval.retrievers.bm25 import tokenize  # same tokenizer BM25 indexes with

# Build title -> full document text from the corpus (gold docs are by title).
corpus = read_parquet_s3(f"{DATA_PREFIX}/corpus.parquet")
doc_text = {r.title: f"{r.title}\n{r.body}" for r in corpus.itertuples()}

def overlap(query, titles):
    q = set(tokenize(query))
    if not q:
        return 0.0
    gold = set()
    for t in titles:
        gold |= set(tokenize(doc_text.get(t, "")))
    return len(q & gold) / len(q)  # fraction of query tokens present in gold docs

final["overlap_orig"] = [overlap(r.original_query, r.evidence_titles) for r in final.itertuples()]
final["overlap_degr"] = [overlap(r.degraded_query, r.evidence_titles) for r in final.itertuples()]

mo, md = final["overlap_orig"].mean(), final["overlap_degr"].mean()
print(f"mean query->gold lexical overlap   original: {mo:.3f}   degraded: {md:.3f}")
print(f"absolute drop: {mo-md:.3f}   relative drop: {(mo-md)/mo*100:.1f}%")
print("\nby question type:")
print(final.groupby("question_type")[["overlap_orig", "overlap_degr"]].mean().round(3).to_string())

# persist the overlap stats alongside the queries for the writeup
buf = io.BytesIO(); final.to_parquet(buf, index=False); buf.seek(0)
s3.put_object(Bucket=BUCKET, Key=OUT_KEY, Body=buf.getvalue())
print(f"\nre-wrote s3://{BUCKET}/{OUT_KEY} with overlap columns")